### Create Data directory

In [1]:
import os
import shutil

In [2]:
if not os.path.exists('.\data'):
    os.mkdir('data')    

### Create CSV from logs

In [ ]:
import re
data_log_tuples = []
with open('./data/access.log', 'r') as logs:
  false_count = 0
  for log in logs.readlines():
    try:
      log_token = re.findall(r'''(\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}) - - \[(.*?)\] \"(.*?)\" (\d{0,3}) (\d{0,4}) \"(.*?)\" \"(.*?)\" \"(.*?)\"''', log)
      data_log_tuples.append(log_token[0])
    except:
      false_count += 1
  else:
    print("Not recognized patterns", false_count)

In [ ]:
import pandas as pd
df = pd.DataFrame(data_log_tuples, columns =['Client IP', 'TIMESTAMP', 'URL Path', 'Response', 'Port', 'URL', 'Device & Browser', '-'])
df.head()

In [ ]:
df['TIMESTAMP'] = pd.to_datetime(df['TIMESTAMP'], format='%d/%b/%Y:%H:%M:%S %z') # converting string to DateTime

In [ ]:
df.to_csv("./data/access.csv") # Saving CSV

### Loading logs csv

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('./data/access.csv', na_values=['-']) # loading csv
df.head()

,Unnamed: 0,Client IP,TIMESTAMP,URL Path,Response,Port,URL,Device & Browser,-
0,0,31.56.96.51,2019-01-22 03:56:16+03:30,GET /image/60844/productModel/200x200 HTTP/1.1,200,5667,https://www.zanbil.ir/m/filter/b113,Mozilla/5.0 (Linux; Android 6.0; ALE-L21 Build...,NaN
1,1,31.56.96.51,2019-01-22 03:56:16+03:30,GET /image/61474/productModel/200x200 HTTP/1.1,200,5379,https://www.zanbil.ir/m/filter/b113,Mozilla/5.0 (Linux; Android 6.0; ALE-L21 Build...,NaN
2,2,40.77.167.129,2019-01-22 03:56:17+03:30,GET /image/14925/productModel/100x100 HTTP/1.1,200,1696,NaN,Mozilla/5.0 (compatible; bingbot/2.0; +http://...,NaN
3,3,40.77.167.129,2019-01-22 03:56:17+03:30,GET /image/23488/productModel/150x150 HTTP/1.1,200,2654,NaN,Mozilla/5.0 (compatible; bingbot/2.0; +http://...,NaN
4,4,40.77.167.129,2019-01-22 03:56:18+03:30,GET /image/45437/productModel/150x150 HTTP/1.1,200,3688,NaN,Mozilla/5.0 (compatible; bingbot/2.0; +http://...,NaN


In [3]:
df.isna().sum() # checking na values

Unnamed: 0                0
Client IP                 0
TIMESTAMP                 0
URL Path                  6
Response                  0
Port                      0
URL                  816667
Device & Browser      13749
-                   7501569
dtype: int64

# LDA

In [4]:
import gensim
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from nltk.stem import WordNetLemmatizer, SnowballStemmer
from nltk.stem.porter import *
import numpy as np
np.random.seed(2018)
import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\harsh\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [5]:
# Creating series from URL Path
data_text = df[['URL Path']]
data_text['index'] = data_text.index
documents = data_text

<ipython-input-5-ef5eb152105f>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_text['index'] = data_text.index


In [6]:
print(len(documents))
print(documents[:5])

7518456
                                         URL Path  index
0  GET /image/60844/productModel/200x200 HTTP/1.1      0
1  GET /image/61474/productModel/200x200 HTTP/1.1      1
2  GET /image/14925/productModel/100x100 HTTP/1.1      2
3  GET /image/23488/productModel/150x150 HTTP/1.1      3
4  GET /image/45437/productModel/150x150 HTTP/1.1      4


In [7]:
# Lammitizing string
def lemmatize_stemming(text):
    return stemmer.stem(WordNetLemmatizer().lemmatize(text, pos='v'))

# Removing stop words
def preprocess(text):
    try:
      result = []
      for token in gensim.utils.simple_preprocess(text):
          if token not in gensim.parsing.preprocessing.STOPWORDS and len(token) > 3:
              result.append(lemmatize_stemming(token))
      return result
    except:
      return 'url'

In [8]:
stemmer = PorterStemmer()

In [9]:
doc_sample = documents[documents['index'] == 4310].values[0][0]
print('original document: ')
words = []
for word in doc_sample.split(' '):
    words.append(word)
print(words)

original document: 
['GET', '/image/%7B%7BbasketItem.id%7D%7D?type=productModel&wh=50x50', 'HTTP/1.1']


In [10]:
print('\n\n tokenized and lemmatized document: ')
print(preprocess(doc_sample))



 tokenized and lemmatized document: 
['imag', 'bbasketitem', 'type', 'productmodel', 'http']


In [11]:
#Preprocess the headline text, saving the results as ‘processed_docs’

processed_docs = documents['URL Path'].map(preprocess)    
processed_docs[:10]

0    [imag, productmodel, http]
1    [imag, productmodel, http]
2    [imag, productmodel, http]
3    [imag, productmodel, http]
4    [imag, productmodel, http]
5    [imag, productmodel, http]
6          [imag, articl, http]
7    [imag, productmodel, http]
8    [imag, productmodel, http]
9    [imag, productmodel, http]
Name: URL Path, dtype: object

### Bag of Words on the Data set

In [12]:
for i, x in enumerate(processed_docs):
  if type(x) != type(list()):
    processed_docs[i] = x.split()


In [13]:
# Create a dictionary from ‘processed_docs’ containing the number of times a word appears in the training set.
dictionary = gensim.corpora.Dictionary(processed_docs)
count = 0
for k, v in dictionary.iteritems():
    print(k, v)
    count += 1
    if count > 10:
        break

0 http
1 imag
2 productmodel
3 articl
4 logo
5 set
6 blog
7 static
8 instagram
9 telegram
10 fwww


In [14]:
# Filter out tokens that appear in
dictionary.filter_extremes(no_below=15, no_above=0.5, keep_n=100000)


In [15]:
#For each document we create a dictionary reporting how many words and how many times those words appear
bow_corpus = [dictionary.doc2bow(doc) for doc in processed_docs]
bow_corpus[4310]

[(0, 1), (63, 1), (64, 1)]

In [16]:
# Preview Bag Of Words for our sample preprocessed document.
bow_doc_4310 = bow_corpus[4310]
for i in range(len(bow_doc_4310)):
    print("Word {} (\"{}\") appears {} time.".format(bow_doc_4310[i][0], 
                                               dictionary[bow_doc_4310[i][0]], 
bow_doc_4310[i][1]))

Word 0 ("productmodel") appears 1 time.
Word 63 ("bbasketitem") appears 1 time.
Word 64 ("type") appears 1 time.


### TF-IDF

In [17]:
'''
Create tf-idf model object using models.TfidfModel on ‘bow_corpus’ and save it to ‘tfidf’, then apply transformation to 
the entire corpus and call it ‘corpus_tfidf’. Finally we preview TF-IDF scores for our first document.
'''

from gensim import corpora, models
tfidf = models.TfidfModel(bow_corpus)
corpus_tfidf = tfidf[bow_corpus]
from pprint import pprint
for doc in corpus_tfidf:
    pprint(doc)
    break

[(0, 1.0)]


In [18]:
# Running LDA using Bag of Words
import pickle

try:
    with open('lda_model.pickle', 'rb') as file:
        lda_model = pickle.load(file)
except:
    lda_model = gensim.models.LdaMulticore(bow_corpus, num_topics=10, id2word=dictionary, passes=2, workers=2)    
    with open('lda_model.pickle', 'wb') as file:
        pickle.dump(lda_model, file, protocol=pickle.HIGHEST_PROTOCOL)

In [19]:
# For each topic, we will explore the words occuring in that topic and its relative weight.
for idx, topic in lda_model.print_topics(-1):
    print('Topic: {} \nWords: {}'.format(idx, topic))

Topic: 0 
Words: 0.268*"logo" + 0.267*"set" + 0.070*"appid" + 0.068*"zanbil" + 0.067*"html" + 0.066*"parentorigin" + 0.066*"helper" + 0.066*"frame" + 0.027*"ampproject" + 0.010*"json"
Topic: 1 
Words: 0.492*"site" + 0.262*"producttypemenu" + 0.070*"enamad" + 0.034*"mobil" + 0.030*"basket" + 0.013*"plu" + 0.012*"addedvalu" + 0.011*"digit" + 0.007*"port" + 0.004*"alert"
Topic: 2 
Words: 0.813*"brand" + 0.068*"filter" + 0.031*"ajaxfilt" + 0.023*"page" + 0.012*"note" + 0.010*"stexist" + 0.007*"consol" + 0.004*"producttypetyp" + 0.004*"skin" + 0.004*"cstexist"
Topic: 3 
Words: 0.163*"product" + 0.144*"icon" + 0.138*"producttyp" + 0.065*"touch" + 0.065*"appl" + 0.056*"galaxi" + 0.039*"provinc" + 0.034*"precompos" + 0.027*"data" + 0.020*"black"
Topic: 4 
Words: 0.994*"productmodel" + 0.001*"content" + 0.001*"photo_" + 0.001*"spin_prod_" + 0.001*"view" + 0.001*"butan" + 0.000*"copi" + 0.000*"shoppingrul" + 0.000*"webfont" + 0.000*"font_awesom"
Topic: 5 
Words: 0.301*"static" + 0.192*"font" + 0

### Running LDA using TF-IDF

In [20]:
try:
    with open('lda_model_tfidf.pickle', 'rb') as file:
        lda_model_tfidf = pickle.load(file)
except:
    lda_model_tfidf = gensim.models.LdaMulticore(corpus_tfidf, num_topics=10, id2word=dictionary, passes=2, workers=4)
    with open('lda_model_tfidf', 'wb') as file:
        pickle.dump(processed_docs, file, protocol=pickle.HIGHEST_PROTOCOL)
        
for idx, topic in lda_model_tfidf.print_topics(-1):
    print('Topic: {} Word: {}'.format(idx, topic))

Topic: 0 Word: 0.342*"telegram" + 0.198*"filter" + 0.151*"static" + 0.084*"prev" + 0.048*"galaxi" + 0.030*"stexist" + 0.028*"page" + 0.027*"core" + 0.021*"black" + 0.013*"plu"
Topic: 1 Word: 0.345*"logo" + 0.344*"set" + 0.061*"exist" + 0.055*"phrase" + 0.029*"static" + 0.028*"jslider" + 0.023*"provinc" + 0.018*"preparesearch" + 0.014*"buypag" + 0.014*"addtozanbil"
Topic: 2 Word: 0.980*"productmodel" + 0.014*"check" + 0.004*"static" + 0.001*"big_sa" + 0.000*"chromstyl" + 0.000*"asu" + 0.000*"bonel" + 0.000*"jcountdown" + 0.000*"vivobook" + 0.000*"medium"
Topic: 3 Word: 0.147*"static" + 0.133*"guarante" + 0.082*"fastdeliveri" + 0.071*"warranti" + 0.071*"instagram" + 0.070*"support" + 0.049*"arrow" + 0.047*"producttyp" + 0.040*"search" + 0.028*"articl"
Topic: 4 Word: 0.250*"producttypemenu" + 0.235*"favicon" + 0.137*"specialsal" + 0.135*"role" + 0.084*"post" + 0.051*"head" + 0.045*"ajaxfilt" + 0.022*"static" + 0.011*"page" + 0.004*"photo_"
Topic: 5 Word: 0.224*"bestpric" + 0.138*"guarante

### Performance evaluation by classifying sample document using LDA Bag of Words model

In [21]:
processed_docs[4310]

['imag', 'bbasketitem', 'type', 'productmodel', 'http']

In [22]:
for index, score in sorted(lda_model[bow_corpus[4310]], key=lambda tup: -1*tup[1]):
    print("\nScore: {}\t \nTopic: {}".format(score, lda_model.print_topic(index, 10)))


Score: 0.27500224113464355	 
Topic: 0.994*"productmodel" + 0.001*"content" + 0.001*"photo_" + 0.001*"spin_prod_" + 0.001*"view" + 0.001*"butan" + 0.000*"copi" + 0.000*"shoppingrul" + 0.000*"webfont" + 0.000*"font_awesom"

Score: 0.2750004529953003	 
Topic: 0.286*"static" + 0.148*"blog" + 0.092*"zanbil" + 0.082*"type" + 0.052*"kharid" + 0.047*"head" + 0.044*"accordion" + 0.042*"icon" + 0.027*"brows" + 0.025*"discountlabel"

Score: 0.2749957740306854	 
Topic: 0.301*"static" + 0.192*"font" + 0.096*"wyekan" + 0.095*"woff" + 0.082*"telegram" + 0.061*"arrow" + 0.048*"search" + 0.027*"exist" + 0.024*"bbasketitem" + 0.021*"categori"

Score: 0.02500021830201149	 
Topic: 0.268*"logo" + 0.267*"set" + 0.070*"appid" + 0.068*"zanbil" + 0.067*"html" + 0.066*"parentorigin" + 0.066*"helper" + 0.066*"frame" + 0.027*"ampproject" + 0.010*"json"

Score: 0.02500021830201149	 
Topic: 0.492*"site" + 0.262*"producttypemenu" + 0.070*"enamad" + 0.034*"mobil" + 0.030*"basket" + 0.013*"plu" + 0.012*"addedvalu" + 

In [23]:
# Performance evaluation by classifying sample document using LDA TF-IDF model.
for index, score in sorted(lda_model_tfidf[bow_corpus[4310]], key=lambda tup: -1*tup[1]):
    print("\nScore: {}\t \nTopic: {}".format(score, lda_model_tfidf.print_topic(index, 10)))


Score: 0.5348837971687317	 
Topic: 0.221*"blog" + 0.151*"product" + 0.093*"static" + 0.089*"type" + 0.070*"bbasketitem" + 0.053*"star" + 0.053*"rati" + 0.028*"discountlabel" + 0.027*"convers" + 0.018*"basket"

Score: 0.2651158571243286	 
Topic: 0.980*"productmodel" + 0.014*"check" + 0.004*"static" + 0.001*"big_sa" + 0.000*"chromstyl" + 0.000*"asu" + 0.000*"bonel" + 0.000*"jcountdown" + 0.000*"vivobook" + 0.000*"medium"

Score: 0.02500004693865776	 
Topic: 0.342*"telegram" + 0.198*"filter" + 0.151*"static" + 0.084*"prev" + 0.048*"galaxi" + 0.030*"stexist" + 0.028*"page" + 0.027*"core" + 0.021*"black" + 0.013*"plu"

Score: 0.02500004693865776	 
Topic: 0.345*"logo" + 0.344*"set" + 0.061*"exist" + 0.055*"phrase" + 0.029*"static" + 0.028*"jslider" + 0.023*"provinc" + 0.018*"preparesearch" + 0.014*"buypag" + 0.014*"addtozanbil"

Score: 0.02500004693865776	 
Topic: 0.147*"static" + 0.133*"guarante" + 0.082*"fastdeliveri" + 0.071*"warranti" + 0.071*"instagram" + 0.070*"support" + 0.049*"arrow

### Testing model on unseen document

In [24]:
unseen_document = '/drive/19nKJL1MQv7y15DhakZZGX86nBRe1SNo_#scrollTo=eli0UIUPnKqJ'
bow_vector = dictionary.doc2bow(preprocess(unseen_document))
for index, score in sorted(lda_model[bow_vector], key=lambda tup: -1*tup[1]):
    print("Score: {}\t Topic: {}".format(score, lda_model.print_topic(index, 5)))

Score: 0.5493140816688538	 Topic: 0.813*"brand" + 0.068*"filter" + 0.031*"ajaxfilt" + 0.023*"page" + 0.012*"note"
Score: 0.05007621645927429	 Topic: 0.268*"logo" + 0.267*"set" + 0.070*"appid" + 0.068*"zanbil" + 0.067*"html"
Score: 0.05007621645927429	 Topic: 0.492*"site" + 0.262*"producttypemenu" + 0.070*"enamad" + 0.034*"mobil" + 0.030*"basket"
Score: 0.05007621645927429	 Topic: 0.163*"product" + 0.144*"icon" + 0.138*"producttyp" + 0.065*"touch" + 0.065*"appl"
Score: 0.05007621645927429	 Topic: 0.994*"productmodel" + 0.001*"content" + 0.001*"photo_" + 0.001*"spin_prod_" + 0.001*"view"
Score: 0.05007621645927429	 Topic: 0.301*"static" + 0.192*"font" + 0.096*"wyekan" + 0.095*"woff" + 0.082*"telegram"
Score: 0.05007621645927429	 Topic: 0.193*"date" + 0.192*"order" + 0.185*"code" + 0.096*"post" + 0.066*"specialsal"
Score: 0.05007621645927429	 Topic: 0.373*"static" + 0.242*"guarante" + 0.056*"favicon" + 0.050*"fastdeliveri" + 0.049*"bestpric"
Score: 0.05007621645927429	 Topic: 0.256*"width